# Пакет ``xml``

Библиотека ElementTree включает инструментальные средства, предназначенные для синтаксического анализа (парсинга) XML-документов c использованием
двух подходов, основанных соответственно на обработке событий в процессе
чтения документа и на объектной модели документа, а также для поиска XML-элементов c помощью выражений XPath и создания или изменения существующих документов.

XML-документы, прошедшие синтаксический анализ, представляются в памяти объектами ElementTree и Element, объединяемыми в единую иерархическую
структуру на основе способа вложения узлов в исходном документе.

## Парсинг

```XML
<?xml version="1.0" encoding="UTF-8"?>
<opml version="1.0">
<head>
    <title>My Podcasts</title>
    <dateCreated>Sat, 06 Aug 2016 15:53:26 GMT</dateCreated>
    <dateModified>Sat, 06 Aug 2016 15:53:26 GMT</dateModified>
</head>
<body>
    <outline text="Non-tech">
        <outline
        text="99% Invisible" type="rss"
        xmlUrl="http://feeds.99percentinvisible.org/
A99percentinvisible"
        htmlUrl="http://99percentinvisible.org" />
    </outline>
    <outline text="Python">
        <outline
            text="Talk Python to Me" type="rss"
            xmlUrl="https://talkpython.fm/episodes/rss"
            htmlUrl="https://talkpython.fm" />
        <outline
            text="Podcast.__init__" type="rss"
            xmlUrl="http://podcastinit.podbean.com/feed/"
            htmlUrl="http://podcastinit.com" />
    </outline>
/body>
/opml>
```

* ``parse(дескриптор открытого файла)`` - синтаксический анализ XML-документа.
*Выход*: объект ``ElementTree``

In [ ]:
from xml.etree import ElementTree
with open('podcasts.opml', 'rt') as f:
    tree = ElementTree.parse(f)
print()

```
<xml.etree.ElementTree.ElementTree object at 0x1013e5630>
```

## Обход дерева узлов

* ``tree.iter()`` - метод для поочередного обхода всех ухлов иерархичекого дерева элементов документа, который возвращает генератор, похволяющий итерироваться по экземляру ElementTree.

In [ ]:
from xml.etree import ElementTree
import pprint
with open('podcasts.opml', 'rt') as f:
    tree = ElementTree.parse(f)

for node in tree.iter():
    print(node.tag)

python3 ElementTree_dump_opml.py\
```
opml
head
title
dateCreated
dateModified
body
outline
outline
outline
outline
outline
```

Чтобы вывести на печать лишь группы имен и URL-адреса подкастов, пропустим все данные в разделе заголовка и выведем лишь атрибуты text и xmlUrl, выполнив поиск соответствующих значений в словаре attrib.\
Аргумент ```'outline'``` метода ``iter()`` задает обработку лишь тех узлов, которые
соответствуют дескрипторам ```outline```.

In [ ]:
from xml.etree import ElementTree

with open('podcasts.opml', 'rt') as f:
    tree = ElementTree.parse(f)
    
for node in tree.iter('outline'):
    name = node.attrib.get('text')
    url = node.attrib.get('xmlUrl')
    if name and url:
        print(' %s' % name)
        print(' %s' % url)
    else:
        print(name)

```
Non-tech
    99% Invisible
        http://feeds.99percentinvisible.org/99percentinvisible
Python
    Talk Python to Me
        https://talkpython.fm/episodes/rss
Podcast.__init__
    http://podcastinit.podbean.com/feed/
```

## Поиск узлов в документе

Обход всего дерева для поиска нужных узлов чреват ошибками. В предыдущем примере мы должны были проверять для каждого узла outline, соответствует ли он группе (узлы, содержащие только атрибут text) или подкасгу
(узлы, содержащие атрибуты text и xmlUrl). Для получения простого списка
URL-адресов потоков без имен и групп логику можно упростить за счет привлечения метода ``findall()`` для поиска узлов c использованием более описательных
характеристик.


In [ ]:
from xml.etree import ElementTree
with open('podcasts.opml', 'rt') as f:
    tree = ElementTree.parse(f)

for node in tree.findall('.//outline*):
    url = node.attrib.get('xmlUrl')
    if url:
        print(url)

```
$ python3 ElementTree_find_feeds__by_tag.py
http://feeds.99percentinvisible.org/99percentinvisible
https://talkpython.fm/episodes/rss
http://podcastinit.podbean.com/feed/
```

Можно воспользоваться тем фактом, что узлы outline имеют только два уровня вложенности. Изменение пути поиска на ``.//outline/outline`` означает, что
цикл будет обрабатывать только узлы outline второго уровня.

In [ ]:
from xml.etree import ElementTree
with open('podcasts.opml', 'rt') as f:
    tree = ElementTree.parse(f)

for node in tree.findall('.//outline/outline'):
    url = node.attrib.get('xmlUrl')
    print(url)

Предполагается, что в исходном документе все узлы outline второго уровня
годержат атрибут xmlURLattribute, ссылающийся на канал подкаста, поэтому
проверку его наличия можно опустить.

```
? python3 ElementTree_find_feeds__by_structure.py
http://feeds.99percentinvisible.org/99percentinvisible
https://talkpython.fm/episodes/rss
http://podcastinit.podbean.com/feed/
```

## Атрибуты узлов

Элементами, возвращаемыми функциями findall () и iter (), являются объекты Element, каждый из которых представляетузел в иерархическом представлении XML-документа. Каждый объект Element имеет атрибуты для доступа к
данным, извлеченным из XML. 

```XML
<?xml version="1.0" encoding="UTF-8"?>
<top>
    <child>Regular text.</child>
    <child_with_tail>Regular text.</child_with_tail>"Tail" text.
    <with_attributes name="value" foo="bar" />
    <entity__expansion attribute="This &#38; That”>
        That &#38; This
    </entity_expansion>
</top>
```

In [2]:
from xml.etree import ElementTree

with open('data.xml', 'rt') as f:
    tree = ElementTree.parse(f)

    node = tree.find('./with_attributes')
print(node.tag)
for name, value in sorted(node.attrib.items()):
    print(' %-4s = "%s"' % (name, value))

IndentationError: unexpected indent (823707846.py, line 2)

```
$ python3 ElementTree_node_attributes.py
with_attributes
foo = "bar"
name = "value"
```

Текстовое содержимое узлов доступно вместе c прикрепленным к ним (tail)
текстом, следующим после закрывающегодескриптора.

In [ ]:
from xml.etree import ElementTree
with open('data.xml', 'rt') as f:
    tree = ElementTree.parse(f)
          
for path in ['./child', './child_with_tail']:
node = tree.find(path)
          print(node.tag)
print(' child node text:', node.text)
print(' and tail text :', node.tail)


```
$ python3 ElementTree_node_text.py
child
child node text: Regular text.
and tail text :
child_with_tail
child node text: Regular text,
and tail text : "Tail" text.
```

Ссылки на XML-сущности, встроенные в документ, преобразуются в соответствующие символы до возврата значений.

In [ ]:
from xml.etree import ElementTree
with open('data.xml', 'rt') as f:
    tree = ElementTree.parse(f)
    
node = tree.find('entity_expansion')
print(node.tag)
print(' in attribute:', node.attrib['attribute'])
print(' in text :', node.text.strip())


```
entity_expansion
in attribute: This & That
in text : That & This
```

## Отслеживание событий в процессе анализа документа

Второй API, предназначенный для обработки XML-документов, основан на
обработке событий. Когда встречается открывающий дескриптор, анализатор
генерирует событие start, а когда закрывающий — событие end. На этапе синтаксического анализа данные могут извлекаться путем итерирования по потоку
событий, что удобно, если после этого не требуется манипулировать сразу всем
документом, а проанализированный документ не должен храниться в памяти
целиком.

**Возможные типы событий:**
* ``start`` - Встретился новый дескриптор. Обрабатывается закрывающая угловая
скобка дескриптора, но не его содержимое.
* ``end`` - Обрабатывается закрывающая угловая скобка закрывающего дескриптора.
К этому моменту все дочерние узлы уже были обработаны.
* ``start-ns`` - Начало объявления пространства имен.

* ``end-ns`` - Конец объявления пространства имен

* ``.iterparse()`` - ) возвращает итерируемый объект, который вырабатывает
кортежи, содержащие имя события и узел, запустивший это событие.


In [ ]:
from xml.etree.ElementTree import iterparse
depth = 0
prefix_width = 8
prefix_dots = ’.' * prefix_width
line_template = ''.join([
'{prefix:<0.{prefix_len}}',
’{event:<8}’,
'{suffix:<{suffix__len}} ’,
’{node.tag:<12} ’,
'{node_id}’,
])
EVENT_NAMES= ['start', ’end’, 'start-ns', ’end-ns’]
for (event, node) in iterparse(*podcasts.opml', EVENT_NAMES):
if event == 'end':
depth -= 1
prefix_len = depth * 2
print(line_template.format(
prefix=prefix_dots,
prefix_len=prefix_len,
suffix=’’,
suffix_len=(prefix_width - prefix_len),
node=node,
node_id=id(node),
event=event,
))
if event == ’start’:
depth += 1


## Задача

Вам дано описание пирамиды из кубиков в формате XML.
Кубики могут быть трех цветов: красный (red), зеленый (green) и синий (blue﻿).
Для каждого кубика известны его цвет, и известны кубики, расположенные прямо под ним.\
Введем понятие ценности для кубиков. Самый верхний кубик, соответствующий корню XML документа имеет ценность 1. Кубики, расположенные прямо под ним, имеют ценность 2. Кубики, расположенные прямо под нижележащими кубиками, имеют ценность 3. И т. д.

Ценность цвета равна сумме ценностей всех кубиков этого цвета.

Выведите через пробел три числа: ценности красного, зеленого и синего цветов.

```XML
<cube color="blue">
  <cube color="red">
    <cube color="green">
    </cube>
  </cube>
  <cube color="red">
  </cube>
</cube>
```
**Вывод:** 4 3 1

In [90]:
from xml.etree import ElementTree
tree = ElementTree.fromstring('<cube color="blue"><cube color="red"><cube color="green"></cube></cube><cube color="red"></cube></cube>')
i=0
color_book={'red':0, 'green':0, 'blue':0}
while True:
    t=[]
    for node in tree.findall(f".{'/cube'*i}"):
        t.append(node.attrib['color'])
    if len(t)==0:
        break
    else:
        for color in t:
            color_book[color]+=i+1
    i+=1
    
print(f"{color_book['red']} {color_book['green']} {color_book['blue']}")

4 3 1


<colgroup>
<col style="width: 30%" />
<col style="width: 70%" />
</colgroup>
<thead>
<tr class="row-odd"><th class="head"><p>Syntax</p></th>
<th class="head"><p>Meaning</p></th>
</tr>
</thead>
<tbody>
<tr class="row-even"><td><p><code class="docutils literal notranslate"><span class="pre">tag</span></code></p></td>
<td><p>Selects all child elements with the given tag.
For example, <code class="docutils literal notranslate"><span class="pre">spam</span></code> selects all child elements
named <code class="docutils literal notranslate"><span class="pre">spam</span></code>, and <code class="docutils literal notranslate"><span class="pre">spam/egg</span></code> selects all
grandchildren named <code class="docutils literal notranslate"><span class="pre">egg</span></code> in all children named
<code class="docutils literal notranslate"><span class="pre">spam</span></code>.  <code class="docutils literal notranslate"><span class="pre">{namespace}*</span></code> selects all tags in the
given namespace, <code class="docutils literal notranslate"><span class="pre">{*}spam</span></code> selects tags named
<code class="docutils literal notranslate"><span class="pre">spam</span></code> in any (or no) namespace, and <code class="docutils literal notranslate"><span class="pre">{}*</span></code>
only selects tags that are not in a namespace.</p>
<div class="versionchanged">
<p><span class="versionmodified changed">Changed in version 3.8: </span>Support for star-wildcards was added.</p>
</div>
</td>
</tr>
<tr class="row-odd"><td><p><code class="docutils literal notranslate"><span class="pre">*</span></code></p></td>
<td><p>Selects all child elements, including comments and
processing instructions.  For example, <code class="docutils literal notranslate"><span class="pre">*/egg</span></code>
selects all grandchildren named <code class="docutils literal notranslate"><span class="pre">egg</span></code>.</p></td>
</tr>
<tr class="row-even"><td><p><code class="docutils literal notranslate"><span class="pre">.</span></code></p></td>
<td><p>Selects the current node.  This is mostly useful
at the beginning of the path, to indicate that it’s
a relative path.</p></td>
</tr>
<tr class="row-odd"><td><p><code class="docutils literal notranslate"><span class="pre">//</span></code></p></td>
<td><p>Selects all subelements, on all levels beneath the
current  element.  For example, <code class="docutils literal notranslate"><span class="pre">.//egg</span></code> selects
all <code class="docutils literal notranslate"><span class="pre">egg</span></code> elements in the entire tree.</p></td>
</tr>
<tr class="row-even"><td><p><code class="docutils literal notranslate"><span class="pre">..</span></code></p></td>
<td><p>Selects the parent element.  Returns <code class="docutils literal notranslate"><span class="pre">None</span></code> if the
path attempts to reach the ancestors of the start
element (the element <code class="docutils literal notranslate"><span class="pre">find</span></code> was called on).</p></td>
</tr>
<tr class="row-odd"><td><p><code class="docutils literal notranslate"><span class="pre">[&#64;attrib]</span></code></p></td>
<td><p>Selects all elements that have the given attribute.</p></td>
</tr>
<tr class="row-even"><td><p><code class="docutils literal notranslate"><span class="pre">[&#64;attrib='value']</span></code></p></td>
<td><p>Selects all elements for which the given attribute
has the given value.  The value cannot contain
quotes.</p></td>
</tr>
<tr class="row-odd"><td><p><code class="docutils literal notranslate"><span class="pre">[&#64;attrib!='value']</span></code></p></td>
<td><p>Selects all elements for which the given attribute
does not have the given value. The value cannot
contain quotes.</p>
<div class="versionadded">
<p><span class="versionmodified added">New in version 3.10.</span></p>
</div>
</td>
</tr>
<tr class="row-even"><td><p><code class="docutils literal notranslate"><span class="pre">[tag]</span></code></p></td>
<td><p>Selects all elements that have a child named
<code class="docutils literal notranslate"><span class="pre">tag</span></code>.  Only immediate children are supported.</p></td>
</tr>
<tr class="row-odd"><td><p><code class="docutils literal notranslate"><span class="pre">[.='text']</span></code></p></td>
<td><p>Selects all elements whose complete text content,
including descendants, equals the given <code class="docutils literal notranslate"><span class="pre">text</span></code>.</p>
<div class="versionadded">
<p><span class="versionmodified added">New in version 3.7.</span></p>
</div>
</td>
</tr>
<tr class="row-even"><td><p><code class="docutils literal notranslate"><span class="pre">[.!='text']</span></code></p></td>
<td><p>Selects all elements whose complete text content,
including descendants, does not equal the given
<code class="docutils literal notranslate"><span class="pre">text</span></code>.</p>
<div class="versionadded">
<p><span class="versionmodified added">New in version 3.10.</span></p>
</div>
</td>
</tr>
<tr class="row-odd"><td><p><code class="docutils literal notranslate"><span class="pre">[tag='text']</span></code></p></td>
<td><p>Selects all elements that have a child named
<code class="docutils literal notranslate"><span class="pre">tag</span></code> whose complete text content, including
descendants, equals the given <code class="docutils literal notranslate"><span class="pre">text</span></code>.</p></td>
</tr>
<tr class="row-even"><td><p><code class="docutils literal notranslate"><span class="pre">[tag!='text']</span></code></p></td>
<td><p>Selects all elements that have a child named
<code class="docutils literal notranslate"><span class="pre">tag</span></code> whose complete text content, including
descendants, does not equal the given <code class="docutils literal notranslate"><span class="pre">text</span></code>.</p>
<div class="versionadded">
<p><span class="versionmodified added">New in version 3.10.</span></p>
</div>
</td>
</tr>
<tr class="row-odd"><td><p><code class="docutils literal notranslate"><span class="pre">[position]</span></code></p></td>
<td><p>Selects all elements that are located at the given
position.  The position can be either an integer
(1 is the first position), the expression <code class="docutils literal notranslate"><span class="pre">last()</span></code>
(for the last position), or a position relative to
the last position (e.g. <code class="docutils literal notranslate"><span class="pre">last()-1</span></code>).</p></td>
</tr>
</tbody>
</table>

In [ ]:
# Top-level elements
root.findall(".")

# All 'neighbor' grand-children of 'country' children of the top-level
# elements
root.findall("./country/neighbor")

# Nodes with name='Singapore' that have a 'year' child
root.findall(".//year/..[@name='Singapore']")

# 'year' nodes that are children of nodes with name='Singapore'
root.findall(".//*[@name='Singapore']/year")

# All 'neighbor' nodes that are the second child of their parent
root.findall(".//neighbor[2]")